In [14]:
import os 
from pathlib import Path

import certifi 
from dotenv import load_dotenv
from langchain_core import prompts
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

from langchain.tools import tool
import requests

In [15]:
from langchain_classic.agents import create_react_agent, AgentExecutor


In [32]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Search for .env in current folder, research folder, or project root
dotenv_path = None
for p in [Path(".env"), Path("research/.env"), Path("../.env")]:
    if p.exists():
        dotenv_path = p.resolve()
        break

load_dotenv(dotenv_path=dotenv_path, override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

print("Loaded from:", dotenv_path)
print("GEMINI_API_KEY loaded:", bool(GEMINI_API_KEY))
print("WEATHERSTACK_API_KEY loaded:", bool(WEATHERSTACK_API_KEY))


Loaded from: D:\Anshul\college program\Python\projects\Lang-Chain\research\.env
GEMINI_API_KEY loaded: True
WEATHERSTACK_API_KEY loaded: True


In [ ]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [21]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [25]:

template = """You are a weather assistant. Your only job is to report the CURRENT weather for the city or cities the user enters.

You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(template)


In [26]:
tools = [get_weather_data]

In [27]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [28]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [35]:
response = agent_executor.invoke({
    "input": (
        
        " Weather in Delhi, India."
    )
})



> Entering new AgentExecutor chain...
Action: get_weather_data
Action Input: Delhi, IndiaCity: Delhi, India
Temperature: 26°C
Weather: Clear
Humidity: 45%I now know the final answer
Final Answer: The current weather in Delhi, India is Clear with a temperature of 26°C and a humidity level of 45%.

> Finished chain.
